In [1]:
from pathlib import Path

import pandas as pd
from pandas.io.formats.style import Styler

HOUR_LABELS = [
    "12-1am", "1-2am", "2-3am", "3-4am", "4-5am", "5-6am",
    "6-7am", "7-8am", "8-9am", "9-10am", "10-11am", "11-12pm",
    "12-1pm", "1-2pm", "2-3pm", "3-4pm", "4-5pm", "5-6pm",
    "6-7pm", "7-8pm", "8-9pm", "9-10pm", "10-11pm", "11-12am",
]

COLOR_MAP = {
    1: "#dfeef8",
    2: "#e9dcc6",
    3: "#f7b267",
    4: "#d9534f",
}

LEGEND_TEXT = "Legend: 1 = light blue, 2 = light brown, 3 = orange, 4 = red. Additional notes: lane openings increase with congestion pressure and represent the access-window intensity used for the operational review."
hour_to_pm = {x: i for i, x in enumerate(range(13, 25), start=1)}


In [6]:
def format_hour(x_str, idx):
    x = int(x_str)
    res = x_str
    if 0 < x < 13:
        res = x_str if idx == 0 else f"{x_str}am"
    elif x == 0:
        res = '12' if idx == 0 else "12pm"
    else:
        res = str(hour_to_pm[x]) if idx == 0 else f"{hour_to_pm[x]}pm"
    return res

def _safe_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def lane_closure_logic(x):
    return x.quantile(0.85)

def _load_matrix_df() -> pd.DataFrame:
    workbook_path = "./assets/qld-traffic-data-average-by-hour-by-day-2025.xlsx"
    df = pd.read_excel(workbook_path)
    df = df.head(5000)
    df.columns = [str(col).strip() for col in df.columns]
    if {"ROAD_NAME", "GAZETTAL_DIRECTION", "HOUR"}.issubset(set(df.columns)):
        return df
    return pd.DataFrame()

In [11]:
def _build_matrix_group(street_name: str, direction: str, df: pd.DataFrame) -> pd.DataFrame:
    day_names = ["Weekday", "SAT", "SUN"]
    weekdays = ['MON', 'TUE', 'WED', 'THU', 'FRI']
    weekends = day_names[1:]

    HOUR_LABELS = df['HOUR'].unique()
    HOUR_IDS = [
        f"hour_{'_'.join(format_hour(x, idx) for idx, x in enumerate(label.split(' to ')))}" 
        for label in HOUR_LABELS
    ]
    if "ROAD_NAME" in df.columns and "GAZETTAL_DIRECTION" in df.columns:
        filtered = df[(df["ROAD_NAME"] == street_name) & (df["GAZETTAL_DIRECTION"] == direction)].copy()
        if filtered.empty:
            rows = []
            for day_name in ["Weekday", "Saturday", "Sunday"]:
                record = {"Direction": direction, "Day-HR": day_name}
                for hour in HOUR_LABELS:
                    record[hour] = 1
                rows.append(record)
            return pd.DataFrame(rows)
        rows = []
        for day in day_names:
            if "HOUR" in filtered:
                filtered = filtered.set_index('HOUR')
            values = filtered.loc[:, [day] if day in weekends else weekdays].apply(lane_closure_logic, axis=1)
            # values = row["VALUE"] if isinstance(row["VALUE"], list) else [1, 2, 3]
            record = {"Direction": direction, "Day-HR": day}
            for hour_id, label in zip(HOUR_IDS, HOUR_LABELS):
                if label in values:
                    record[hour_id] = round(values.loc[label], 2)
            rows.append(record)
        temp = pd.DataFrame(rows)
        return temp
    temp = pd.DataFrame({"Direction": direction, "Day-HR": ["Weekday", "Saturday", "Sunday"]})
    return temp

In [8]:
df = _load_matrix_df()

print(df)
direction_values = sorted(set(df["GAZETTAL_DIRECTION"].dropna().tolist())) if "GAZETTAL_DIRECTION" in df.columns else ["WB", "EB"]

if not street_names:
    street_names = ["Aster Road"]
if not direction_values:
    direction_values = ["WB", "EB"]
print(street_names)

      SITE_ID                       DESCRIPTION  SITE_DISTANCE   LONGITUDE  \
0       10007       900m west of Marble Arch Pl           2.13  153.345772   
1       10007       900m west of Marble Arch Pl           2.13  153.345772   
2       10007       900m west of Marble Arch Pl           2.13  153.345772   
3       10007       900m west of Marble Arch Pl           2.13  153.345772   
4       10007       900m west of Marble Arch Pl           2.13  153.345772   
...       ...                               ...            ...         ...   
4995    20057  West of Rd 483-Gympie Brooloo Rd           0.65  152.694407   
4996    20057  West of Rd 483-Gympie Brooloo Rd           0.65  152.694407   
4997    20057  West of Rd 483-Gympie Brooloo Rd           0.65  152.694407   
4998    20057  West of Rd 483-Gympie Brooloo Rd           0.65  152.694407   
4999    20057  West of Rd 483-Gympie Brooloo Rd           0.65  152.694407   

       LATITUDE ROAD_SECTION_ID                                

In [27]:
street_names = sorted(set(df["ROAD_NAME"].dropna().tolist())) if "ROAD_NAME" in df.columns else ["Aster Road"]
for street_name in street_names[4:8]:
    matrix = pd.DataFrame()
    left_matrix = _build_matrix_group(street_name, direction_values[0], df)
    right_matrix = _build_matrix_group(street_name, direction_values[1] if len(direction_values) > 1 else direction_values[0], df)

    matrix = pd.concat([left_matrix, right_matrix], ignore_index=True)
    print(f"{street_name} mat \n{matrix}")

BEECHMONT ROAD mat 
          Direction   Day-HR  hour_12_1am  hour_1_2am  hour_2_3am  hour_3_4am  \
0  AGAINST GAZETTAL  Weekday          3.0         3.4         3.0         7.4   
1  AGAINST GAZETTAL      SAT          3.0         1.0         2.0         5.0   
2  AGAINST GAZETTAL      SUN          5.0         1.0         5.0         4.0   
3          GAZETTAL  Weekday          5.4         2.4         1.0         3.4   
4          GAZETTAL      SAT          3.0         4.0         3.0         2.0   
5          GAZETTAL      SUN          5.0         5.0         2.0         3.0   

   hour_4_5am  hour_5_6am  hour_6_7am  hour_7_8am  ...  hour_2_3pm  \
0        28.8        85.8       138.0       169.8  ...        90.4   
1         8.0        20.0        28.0        78.0  ...        86.0   
2         8.0        13.0        24.0        45.0  ...       110.0   
3         2.4        15.4        60.4        58.8  ...       105.2   
4         1.0         7.0        17.0        43.0  ...       1

In [28]:
street_names = ["BEAUDESERT - BEENLEIGH ROAD"]
for street_name in street_names:
    left_matrix = _build_matrix_group(street_name, direction_values[0], df)
    right_matrix = _build_matrix_group(street_name, direction_values[1] if len(direction_values) > 1 else direction_values[0], df)

    matrix = pd.concat([left_matrix, right_matrix], ignore_index=True)
    print(f"mat \n{matrix}")

mat 
          Direction   Day-HR  \
0  AGAINST GAZETTAL  Weekday   
1  AGAINST GAZETTAL      SAT   
2  AGAINST GAZETTAL      SUN   
3          GAZETTAL  Weekday   
4          GAZETTAL      SAT   
5          GAZETTAL      SUN   

                                         hour_12_1am  \
0  HOUR
0 to 1    13.4
0 to 1    18.8
0 to 1    6...   
1  HOUR
0 to 1    18.0
0 to 1    33.0
0 to 1    8...   
2  HOUR
0 to 1    18.0
0 to 1    41.0
0 to 1    7...   
3  HOUR
0 to 1     9.4
0 to 1    14.4
0 to 1    2...   
4  HOUR
0 to 1    11.0
0 to 1    17.0
0 to 1    5...   
5  HOUR
0 to 1    12.0
0 to 1    19.0
0 to 1    6...   

                                          hour_1_2am  \
0  HOUR
1 to 2     9.0
1 to 2    20.2
1 to 2    4...   
1  HOUR
1 to 2    11.0
1 to 2    21.0
1 to 2    5...   
2  HOUR
1 to 2    10.0
1 to 2    18.0
1 to 2    6...   
3  HOUR
1 to 2     8.0
1 to 2    12.4
1 to 2    2...   
4  HOUR
1 to 2     7.0
1 to 2    13.0
1 to 2    3...   
5  HOUR
1 to 2     7.0
1 to 2    16.0
1 t

In [17]:
day_names = ["Weekday", "SAT", "SUN"]
weekdays = ['MON', 'TUE', 'WED', 'THU', 'FRI']
weekends = day_names[1:]

HOUR_LABELS = df['HOUR'].unique()
HOUR_IDS = [
    f"hour_{'_'.join(format_hour(x, idx) for idx, x in enumerate(label.split(' to ')))}" 
    for label in HOUR_LABELS
]
if "ROAD_NAME" in df.columns and "GAZETTAL_DIRECTION" in df.columns:
    filtered = df[(df["ROAD_NAME"] == street_name) & (df["GAZETTAL_DIRECTION"] == direction_values[0])].copy()
filtered

,SITE_ID,DESCRIPTION,SITE_DISTANCE,LONGITUDE,LATITUDE,ROAD_SECTION_ID,ROAD_NAME,TDIST_START,TDIST_END,GAZETTAL_DIRECTION,...,THU,FRI,SAT,SUN,WEEKDAY_AVERAGE,WEEKEND_AVERAGE,DISTRICT,DISTRICT_NAME,REGION,REGION_NAME
48,10010,100m north of Tremayne Rd Intersection,5.66,153.080048,-27.927960,203,BEAUDESERT - BEENLEIGH ROAD,0.00,10.03,AGAINST GAZETTAL,...,12.0,14.0,18.0,18.0,12,18.0,410,South Coast,310,South Coast
49,10010,100m north of Tremayne Rd Intersection,5.66,153.080048,-27.927960,203,BEAUDESERT - BEENLEIGH ROAD,0.00,10.03,AGAINST GAZETTAL,...,9.0,9.0,11.0,10.0,8,11.0,410,South Coast,310,South Coast
50,10010,100m north of Tremayne Rd Intersection,5.66,153.080048,-27.927960,203,BEAUDESERT - BEENLEIGH ROAD,0.00,10.03,AGAINST GAZETTAL,...,9.0,10.0,8.0,7.0,9,8.0,410,South Coast,310,South Coast
51,10010,100m north of Tremayne Rd Intersection,5.66,153.080048,-27.927960,203,BEAUDESERT - BEENLEIGH ROAD,0.00,10.03,AGAINST GAZETTAL,...,14.0,15.0,10.0,7.0,14,9.0,410,South Coast,310,South Coast
52,10010,100m north of Tremayne Rd Intersection,5.66,153.080048,-27.927960,203,BEAUDESERT - BEENLEIGH ROAD,0.00,10.03,AGAINST GAZETTAL,...,40.0,38.0,25.0,15.0,39,20.0,410,South Coast,310,South Coast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2936,11735,150m north of Tallagandra Avenue,35.05,153.193692,-27.724918,203,BEAUDESERT - BEENLEIGH ROAD,31.66,35.48,AGAINST GAZETTAL,...,538.0,506.0,394.0,266.0,469,330.0,410,South Coast,310,South Coast
2937,11735,150m north of Tallagandra Avenue,35.05,153.193692,-27.724918,203,BEAUDESERT - BEENLEIGH ROAD,31.66,35.48,AGAINST GAZETTAL,...,432.0,431.0,342.0,115.0,356,228.0,410,South Coast,310,South Coast
2938,11735,150m north of Tallagandra Avenue,35.05,153.193692,-27.724918,203,BEAUDESERT - BEENLEIGH ROAD,31.66,35.48,AGAINST GAZETTAL,...,336.0,350.0,334.0,86.0,271,210.0,410,South Coast,310,South Coast
2939,11735,150m north of Tallagandra Avenue,35.05,153.193692,-27.724918,203,BEAUDESERT - BEENLEIGH ROAD,31.66,35.48,AGAINST GAZETTAL,...,170.0,257.0,263.0,62.0,177,162.0,410,South Coast,310,South Coast


In [20]:
direction = direction_values[0]
rows = []
if filtered.empty:
    rows = []
    for day_name in ["Weekday", "Saturday", "Sunday"]:
        record = {"Direction": direction, "Day-HR": day_name}
        for hour in HOUR_LABELS:
            record[hour] = 1
        rows.append(record)
    temp = pd.DataFrame(rows)
print(rows)
rows = []
for day in day_names:
    print(f"day = {day}")
    if "HOUR" in filtered:
        filtered = filtered.set_index('HOUR')
    values = filtered.loc[:, [day] if day in weekends else weekdays].apply(lane_closure_logic, axis=1)
    print(f"values = {values}")
    record = {"Direction": direction, "Day-HR": day}
    for hour_id, label in zip(HOUR_IDS, HOUR_LABELS):
        if label in values:
            record[hour_id] = round(values.loc[label], 2)
    rows.append(record)
    print(f"NEW\n{record}")
temp = pd.DataFrame(rows)


[]
day = Weekday
values = HOUR
0 to 1       13.4
1 to 2        9.0
2 to 3        9.4
3 to 4       14.4
4 to 5       40.0
            ...  
19 to 20    518.8
20 to 21    431.4
21 to 22    341.6
22 to 23    216.2
23 to 24    124.6
Length: 72, dtype: float64
NEW
{'Direction': 'AGAINST GAZETTAL', 'Day-HR': 'Weekday', 'hour_12_1am': HOUR
0 to 1    13.4
0 to 1    18.8
0 to 1    60.8
dtype: float64, 'hour_1_2am': HOUR
1 to 2     9.0
1 to 2    20.2
1 to 2    40.2
dtype: float64, 'hour_2_3am': HOUR
2 to 3     9.4
2 to 3    16.8
2 to 3    27.0
dtype: float64, 'hour_3_4am': HOUR
3 to 4    14.4
3 to 4    18.6
3 to 4    36.6
dtype: float64, 'hour_4_5am': HOUR
4 to 5    40.0
4 to 5    37.0
4 to 5    86.0
dtype: float64, 'hour_5_6am': HOUR
5 to 6    174.8
5 to 6    194.2
5 to 6    380.8
dtype: float64, 'hour_6_7am': HOUR
6 to 7    273.4
6 to 7    365.4
6 to 7    589.8
dtype: float64, 'hour_7_8am': HOUR
7 to 8    353.8
7 to 8    335.0
7 to 8    575.4
dtype: float64, 'hour_8_9am': HOUR
8 to 9    359.2
